In [1]:
import os
import pandas as pd
import xarray as xr
import numpy as np

In [11]:
# Make sure logged in
import copernicusmarine
copernicusmarine.login()

INFO - 2026-03-17T18:59:30Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register


Copernicus Marine username:

  eholmes


Copernicus Marine password:

  ········


INFO - 2026-03-17T19:00:06Z - Credentials file stored in /home/jovyan/.copernicusmarine/.copernicusmarine-credentials.


True

In [2]:
def clean_dataset(dataset):
    """
    Rename dimensions with standard names and sort values if appropriate
    """
    # Rename dimensions if incorrect
    for coordinate in dataset.coords:
       if coordinate=='lon':
           dataset = dataset.rename({'lon': 'longitude'})
       if coordinate=='lat':
           dataset = dataset.rename({'lat': 'latitude'})
       if coordinate=='DEPTH':
           dataset = dataset.rename({'lat': 'depth'})
          
    # Sort correctly lon/lat if they were inverted
    for dimension in ["latitude","longitude"]:
        coordinates = dataset[dimension].values
        if (coordinates[0] >= coordinates[:-1]).all():
            dataset = dataset.sortby(dimension, ascending=True)
        
    return dataset

In [4]:
import pandas as pd
url = (
    "https://raw.githubusercontent.com/"
    "fish-pace/point-collocation/main/"
    "examples/fixtures/points.csv"
)
df_points = pd.read_csv(url)
print(len(df_points))

# Let's add on our own pc_id column
df_points = df_points.reset_index(drop=True)
df_points["pc_id"] = df_points.index + 1
df_points["pc_label"] = "pace_" + df_points["pc_id"].astype(str)
#df_points = clean_dataset(df_points)
df_points.head()

595


,lat,lon,date,pc_id,pc_label
0,27.3835,-82.7375,2024-06-13,1,pace_1
1,27.1190,-82.7125,2024-06-14,2,pace_2
2,26.9435,-82.8170,2024-06-14,3,pace_3
3,26.6875,-82.8065,2024-06-14,4,pace_4
4,26.6675,-82.6455,2024-06-14,5,pace_5


In [9]:
import pandas as pd
import numpy as np

# number of points
n = 100

# create dataframe
input_dataframe = pd.DataFrame({
    "LATITUDE": np.random.uniform(40, 42, n),
    "LONGITUDE": np.random.uniform(4, 8, n),
    "START": pd.to_datetime("2020-01-01"),
    "END": pd.to_datetime("2020-12-31"),
    "DEPTH": np.random.uniform(0, 10, n),
    "TIME": pd.to_datetime("2020-01-01")
})

print(input_dataframe.head())

    LATITUDE  LONGITUDE      START        END     DEPTH       TIME
0  41.407793   5.053561 2020-01-01 2020-12-31  7.680988 2020-01-01
1  41.404007   4.419560 2020-01-01 2020-12-31  8.197601 2020-01-01
2  40.903891   5.572079 2020-01-01 2020-12-31  9.495776 2020-01-01
3  40.969383   5.708267 2020-01-01 2020-12-31  7.677003 2020-01-01
4  40.955243   5.180251 2020-01-01 2020-12-31  1.556488 2020-01-01


In [7]:
# List of Copernicus Marine datasets (datasetIDs)
list_datasets = [
    'cmems_mod_med_phy-cur_my_4.2km_P1D-m',
]

# Output result names 
output_names = [
    'current_006_004',
]

# Variables to download
variables = ['uo','vo']

# Output directory (will be created if doesn't exist)
output_dir = "Dataframes/"

# SINGLE DATES ONLY: Set to 'True' if you want to keep depth/lat/lon/time from dataset (for checking purpose)
include_dataset_columns = True

# TIMESERIES ONLY: Set to 'True' to save output files in NetCDF format instead of CSV
save_as_netcdf = False


In [10]:
# Create directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

for dataset_id, output_name in zip(list_datasets, output_names):

    # Read dataset via Toolbox
    dataset = copernicusmarine.open_dataset(dataset_id=dataset_id, chunk_size_limit=0) # chunk size limit set to 0 for single time steps

    # Clean dataset
    dataset = clean_dataset(dataset)

    # Select only requested variables first (important for correct depth detection)
    selected_variables_dataset = dataset[variables]

    # Copy input dataframe
    input_points_dataframe = input_dataframe.copy()

    # Build vectorized indexers
    time_points = xr.DataArray(pd.to_datetime(input_points_dataframe["TIME"]).to_numpy(), dims="points")
    latitude_points = xr.DataArray(input_points_dataframe["LATITUDE"].to_numpy(dtype=float), dims="points")
    longitude_points = xr.DataArray(input_points_dataframe["LONGITUDE"].to_numpy(dtype=float), dims="points")

    selection_indexers = {
        "time": time_points,
        "latitude": latitude_points,
        "longitude": longitude_points,
    }

    # Depth detection must be done on the selected variables dataset (not the full dataset)
    dataset_has_depth = "depth" in selected_variables_dataset.dims
    dataframe_has_depth = "DEPTH" in input_points_dataframe.columns and input_points_dataframe["DEPTH"].notna().any()

    # Download data for 3D variables
    if dataset_has_depth:
        if dataframe_has_depth:
            depth_points = xr.DataArray(input_points_dataframe["DEPTH"].to_numpy(dtype=float), dims="points")
            selection_indexers["depth"] = depth_points
            selected_points_dataset = selected_variables_dataset.sel(**selection_indexers, method="nearest")
        else:
            selected_points_dataset = selected_variables_dataset.isel(depth=0).sel(**selection_indexers, method="nearest")
    else:
        # Download data for 2D variables
        selected_points_dataset = selected_variables_dataset.sel(**selection_indexers, method="nearest")

    # Convert dataset to dataframe
    downloaded_variables_dataframe = selected_points_dataset.to_dataframe().reset_index(drop=True)

    # Identify index-like columns coming from xarray
    index_like_columns = [column for column in ("time", "latitude", "longitude", "depth") if column in downloaded_variables_dataframe.columns]

    # Keep depth/lat/lon/time from the dataset only if requested
    if not include_dataset_columns:
        downloaded_variables_dataframe = downloaded_variables_dataframe.drop(columns=index_like_columns, errors="ignore")

    # Add variables to input dataframe
    output_dataframe = pd.concat(
        [input_points_dataframe.reset_index(drop=True), downloaded_variables_dataframe.reset_index(drop=True)],
        axis=1,
    )

    # Save final dataframe with downloaded variables
    output_csv_path = os.path.join(output_dir, f"{output_name}_temporal_points.csv")
    output_dataframe.to_csv(output_csv_path, index=False)

print("Download completed!")

INFO - 2026-03-17T18:33:05Z - Selected dataset version: "202511"
INFO - 2026-03-17T18:33:05Z - Selected dataset part: "default"
INFO - 2026-03-17T18:33:05Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register


Copernicus Marine username:

  eholmes


Copernicus Marine password:

  ········


Download completed!
